# Demand Forecasting — Linear Regression with MLflow

Reads the complete engineered dataset from the Databricks Feature Store (`mle.features.product_city_demand_features`),
splits chronologically by `transaction_date` into train/validation/test (64%/16%/20%), trains a linear regression
model with scikit-learn, and tracks the full experiment in MLflow. All preprocessing (imputation, scaling,
one-hot encoding) is encapsulated in an sklearn `Pipeline` logged as a single deployable model artifact.

**Split strategy:** Chronological by `transaction_date` — training on earlier dates, validating on middle dates,
testing on the latest dates. This mirrors real-world demand forecasting where we predict future demand from
past patterns, avoiding look-ahead bias.

In [0]:
import mlflow
import pandas as pd
import numpy as np

# ── Read from Feature Store ──────────────────────────────────────────
FEATURE_TABLE = "mle.features.product_city_demand_features"
print(f"Reading from Feature Store: {FEATURE_TABLE}")
df = spark.table(FEATURE_TABLE).toPandas()
print(f"Shape: {df.shape}")

# Convert booleans to int for sklearn
for col in ['is_weekend']:
    df[col] = df[col].astype(int)

# Ensure transaction_date is datetime
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# Sort chronologically for time-based split
df = df.sort_values('transaction_date').reset_index(drop=True)

# ── Define target and features ───────────────────────────────────────
TARGET = 'demand_quantity'
EXCLUDE = ['transaction_date', TARGET]

categorical_features = ['product_name', 'destination_city', 'product_category', 'quantity_unit']
numerical_features = [c for c in df.columns if c not in EXCLUDE + categorical_features]

print(f"\nTarget: {TARGET}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"Total features: {len(categorical_features) + len(numerical_features)}")

# ── Quick EDA ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("EDA SUMMARY")
print(f"{'='*60}")
print(f"Date range: {df['transaction_date'].min().date()} → {df['transaction_date'].max().date()}")
print(f"Total rows: {len(df):,}")
print(f"\nTarget ({TARGET}) statistics:")
print(df[TARGET].describe().round(2))

# Null summary
nulls = df[categorical_features + numerical_features + [TARGET]].isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) > 0:
    print(f"\nNull counts (non-zero only):")
    for c, cnt in nulls.items():
        print(f"  {c:30s}: {cnt:,} ({cnt/len(df)*100:.1f}%)")

# Correlations with target
print(f"\nTop 10 |correlations| with target ({TARGET}):")
corrs = df[numerical_features].corrwith(df[TARGET]).abs().sort_values(ascending=False)
for c, v in corrs.head(10).items():
    print(f"  {c:30s}: {v:.4f}")

# Feature type table
print(f"\n{'Feature':<35} {'Type'}")
print("-" * 50)
for f in categorical_features:
    print(f"{f:<35} categorical/nominal")
for f in numerical_features:
    print(f"{f:<35} numeric")

In [0]:
# ── Chronological split ──────────────────────────────────────────────
# 80% train+val / 20% test  →  then 80% train / 20% val from train+val
n = len(df)
idx_test_start = int(n * 0.80)               # first 80% → train+val
idx_val_start = int(idx_test_start * 0.80)   # first 80% of train+val → train

train_df = df.iloc[:idx_val_start].copy()
val_df   = df.iloc[idx_val_start:idx_test_start].copy()
test_df  = df.iloc[idx_test_start:].copy()

print("CHRONOLOGICAL SPLIT SUMMARY")
print(f"{'='*60}")
print(f"Train: {len(train_df):>6,} rows ({len(train_df)/n*100:.1f}%)  |  {train_df['transaction_date'].min().date()} → {train_df['transaction_date'].max().date()}")
print(f"Val:   {len(val_df):>6,} rows ({len(val_df)/n*100:.1f}%)  |  {val_df['transaction_date'].min().date()} → {val_df['transaction_date'].max().date()}")
print(f"Test:  {len(test_df):>6,} rows ({len(test_df)/n*100:.1f}%)  |  {test_df['transaction_date'].min().date()} → {test_df['transaction_date'].max().date()}")
print(f"{'='*60}")

# Prepare X and y
feature_cols = categorical_features + numerical_features

X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_val,   y_val   = val_df[feature_cols],   val_df[TARGET]
X_test,  y_test  = test_df[feature_cols],  test_df[TARGET]

print(f"\nX_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}  y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}  y_test:  {y_test.shape}")

In [0]:
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from mlflow.models import infer_signature

# ── Build preprocessing + model pipeline ─────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
        ]), numerical_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical_features),
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()),
])

# ── MLflow experiment ─────────────────────────────────────────────────
EXPERIMENT_NAME = "/Users/girija.zirange@bizmetric.com/MLE/demand_forecasting"
mlflow.set_experiment(EXPERIMENT_NAME)


def eval_metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    mask = y_true != 0
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.any() else float('inf')
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}


with mlflow.start_run(run_name="linear_regression_baseline") as run:
    # Train
    model.fit(X_train, y_train)

    # Predict
    y_train_pred = model.predict(X_train)
    y_val_pred   = model.predict(X_val)
    y_test_pred  = model.predict(X_test)

    # Evaluate
    train_m = eval_metrics(y_train, y_train_pred)
    val_m   = eval_metrics(y_val, y_val_pred)
    test_m  = eval_metrics(y_test, y_test_pred)

    # Log parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("split_strategy", "chronological_by_transaction_date")
    mlflow.log_param("train_ratio", 0.64)
    mlflow.log_param("val_ratio", 0.16)
    mlflow.log_param("test_ratio", 0.20)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("val_size", len(X_val))
    mlflow.log_param("test_size", len(X_test))
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("n_categorical", len(categorical_features))
    mlflow.log_param("n_numerical", len(numerical_features))
    mlflow.log_param("imputation_num", "mean")
    mlflow.log_param("imputation_cat", "most_frequent")
    mlflow.log_param("scaling", "StandardScaler")
    mlflow.log_param("encoding", "OneHotEncoder")
    mlflow.log_param("feature_table", "mle.features.product_city_demand_features")

    # Log metrics
    for prefix, m in [("train", train_m), ("val", val_m), ("test", test_m)]:
        for name, val in m.items():
            mlflow.log_metric(f"{prefix}_{name}", val)

    # Log model with signature
    signature = infer_signature(X_train.head(100), model.predict(X_train.head(100)))
    model_info = mlflow.sklearn.log_model(
        model,
        name="model",
        signature=signature,
        input_example=X_train.head(3),
    )

    run_id = run.info.run_id
    model_uri = model_info.model_uri

    print(f"MLflow Run ID:  {run_id}")
    print(f"Model URI:      {model_uri}")
    print(f"Experiment:     {EXPERIMENT_NAME}")
    print(f"\n{'='*60}")
    print("MODEL PERFORMANCE")
    print(f"{'='*60}")
    print(f"{'Metric':<10} {'Train':>15} {'Val':>15} {'Test':>15}")
    print("-" * 55)
    for metric in ["rmse", "mae", "r2", "mape"]:
        print(f"{metric:<10} {train_m[metric]:>15.4f} {val_m[metric]:>15.4f} {test_m[metric]:>15.4f}")
    print(f"{'='*60}")

# ── Residual analysis ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

residuals = y_test - y_test_pred
axes[0].hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Test Residuals Distribution')
axes[0].set_xlabel('Residual (Actual \u2212 Predicted)')
axes[0].set_ylabel('Count')
axes[0].text(0.02, 0.98, f"Mean: {residuals.mean():.2f}\nStd: {residuals.std():.2f}",
             transform=axes[0].transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[1].scatter(y_test, y_test_pred, alpha=0.3, s=10, color='steelblue')
lim = max(y_test.max(), y_test_pred.max())
axes[1].plot([0, lim], [0, lim], 'r--', linewidth=2)
axes[1].set_title('Test: Predicted vs Actual')
axes[1].set_xlabel('Actual Demand')
axes[1].set_ylabel('Predicted Demand')

plt.tight_layout()
display(fig)
plt.close()